# Enhanced COCO Multi-Label Visual Agent — Training Demo + API Test

This notebook:
1. Downloads the Kaggle COCO multi-label dataset
2. Trains `MultiLabelCOCONet` (from `model.py`) and produces `train_results.png`
3. Saves weights to `coco_multilabel_cnn.pth`
4. Runs the LangGraph agent end-to-end on a sample image
5. Hits the live FastAPI `/enhanced-vision` endpoint to confirm the full pipeline works

## 0. Setup

In [ ]:
!pip install -q -r requirements.txt

## 1. Download dataset (Kaggle / Colab)

In [ ]:
# In Colab, first upload your kaggle.json (API token) via:
# from google.colab import files
# files.upload()  # select kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d shubham2703/coco-dataset-for-multi-label-image-classification
!unzip -oq coco-dataset-for-multi-label-image-classification.zip -d coco_data
!ls coco_data

## 2. Train the CNN (Phase 1)

Adjust `csv_path` / `img_dir` below to match the actual folder layout you get after unzipping
(the Kaggle download's exact filenames can vary by version).

In [ ]:
from model import CocoMultiLabelDataset, train_model

train_ds = CocoMultiLabelDataset(
    csv_path="coco_data/train_labels.csv",
    img_dir="coco_data/train_images",
    augment=True,
)
val_ds = CocoMultiLabelDataset(
    csv_path="coco_data/val_labels.csv",
    img_dir="coco_data/val_images",
    classes=train_ds.classes,
)

print(f"Classes ({len(train_ds.classes)}):", train_ds.classes)
print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

In [ ]:
model, history = train_model(
    train_ds, val_ds,
    num_classes=len(train_ds.classes),
    class_names=train_ds.classes,
    epochs=15,
    batch_size=32,
    lr=1e-3,
)
print("Final train loss:", history["train_loss"][-1])
print("Final val loss:", history["val_loss"][-1])

In [ ]:
from IPython.display import Image as IPImage
IPImage("train_results.png")

## 3. Run the LangGraph agent directly (Phase 2 sanity check)

Requires `ANTHROPIC_API_KEY` (or your chosen provider's key) set as an environment variable.

In [ ]:
import os
os.environ["ANTHROPIC_API_KEY"] = "sk-..."  # or set this in your shell / Colab secrets

from PIL import Image
from agent_graph import run_agent

test_image = Image.open("test_image.jpg")
result = run_agent(test_image)

print("CNN predictions:", result["cnn_predictions"])
print("\nMultimodal LLM response:\n", result["multimodal_llm_response"])
print("\nFinal enhanced description:\n", result["final_description"])

## 4. Start the FastAPI server (Phase 3)

Run this in a terminal (not in this notebook cell), from the project directory:

```bash
export ANTHROPIC_API_KEY=sk-...
uvicorn app:app --host 0.0.0.0 --port 8000
```

## 5. Test the live API endpoint

In [ ]:
import requests

with open("test_image.jpg", "rb") as f:
    resp = requests.post(
        "http://localhost:8000/enhanced-vision",
        files={"file": ("test_image.jpg", f, "image/jpeg")},
    )

print("Status:", resp.status_code)
import json
print(json.dumps(resp.json(), indent=2))

In [ ]:
# Equivalent curl command for reference:
# curl -X POST "http://localhost:8000/enhanced-vision" -F "file=@test_image.jpg"